<div>
<center><img src="../assets/Flux-logo.svg" width="360"/></center>
</div>

<div style="background:linear-gradient(90deg,#036291 0%,#91C2D8 100%);padding:20px 26px;border-radius:10px;border-left:10px solid #D9A441;margin-top:18px">
<h1 style="margin:0;color:#ffffff">Module 2: Fluence in Kubernetes</h1>
<p style="margin:6px 0 0 0;color:#DCECF4;font-size:15px">We will use kind "Kubernetes in Docker" to install fluence and schedule pods.</p>
<p style="margin:2px 0 0 0;color:#DCECF4;font-size:13px">flux-framework &middot; fluence &middot; kind</p>
</div>


# Welcome

[Fluence](https://github.com/converged-computing/fluence) brings Fluxion, the Flux graph
scheduler, into Kubernetes as a scheduler plugin. This notebook runs it in a
[kind](https://kind.sigs.k8s.io/) cluster. A kind node is a docker container.

Click on <button data-commandLinker-command="terminal:open" data-name="fluence" href="#">this button</button> to create a Terminal and drag it alongside this notebook. You should run commands in that terminal.

```bash
cd ~/tutorial/module2
```

<div class="alert alert-block" style="background-color:#DCECF4;color:#06293D">
<span style="font-weight:600">Notice:</span> If you started the container without <code>-v /var/run/docker.sock:/var/run/docker.sock</code>, docker is not reachable from here. Stop the container and start it again with the mount, the README has the command. Module 1 does not need it.
</div>

## 1. Check docker

```bash
docker info --format '{{.ServerVersion}} on {{.OperatingSystem}}'
kind --version
kubectl version --client
helm version --short
```

The first line is the docker on your machine, not one in this container.


## 2. Bring up the cluster

`kind-up` creates a cluster called `fluence` and makes it reachable from here. By default it is
one control plane and two workers on Kubernetes 1.36, with the alpha gang scheduling pieces
turned on: the `scheduling.k8s.io/v1alpha2` API group and the `GenericWorkload` and
`GangScheduling` feature gates. That config is built into `kind-up`, and
[`manifests/kind-config.yaml`](manifests/kind-config.yaml) is the same file if you want to read it.
fluence places gangs through that native API, so without the gates there are no gang semantics,
and `kind-up` checks for the API once the cluster is up.

```bash
kind-up
```
```console
Creating cluster "fluence" ...
 ✓ Ensuring node image (kindest/node:v1.36.1) 🖼
 ✓ Preparing nodes 📦 📦 📦
 ✓ Writing configuration 📜
 ✓ Starting control-plane 🕹️
 ✓ Installing CNI 🔌
 ✓ Installing StorageClass 💾
 ✓ Joining worker nodes 🚜
Set kubectl context to "kind-fluence"
```

The node is a Docker container, and you can see it from your host running this notebook. The cluster resources will thus mirror your host.

```bash
kubectl get nodes -o jsonpath='{range .items[*]}{.metadata.name}{": cpu="}{.status.allocatable.cpu}{" mem="}{.status.allocatable.memory}{"\n"}{end}'
```

## 3. Install fluence

<div class="alert alert-block" style="background-color:#91C2D8;color:#06293D">
<span style="font-weight:600">Description:</span> fluence is a Kubernetes scheduler plugin that implements the PreFilter, Filter, and PreBind endpoints. At startup it lists the cluster's nodes, builds a Fluxion resource graph from them, and from then on every pod that says <code>schedulerName: fluence</code> is placed by matching a Fluxion jobspec against that graph instead of by the default scheduler.
</div>

Let's install fluence from [`manifests/fluence.yaml`](manifests/fluence.yaml), and right after it
the resources it should model, [`manifests/fluence-resources.yaml`](manifests/fluence-resources.yaml).
The order matters only in that the scheduler and the webhook read the resources config when they
start, so applying it now means nothing has to be restarted later. Section 5 explains what is in it.

```bash
kubectl apply -f manifests/fluence.yaml
kubectl apply -f manifests/fluence-resources.yaml
kubectl rollout status -n kube-system deployment/fluence --timeout=180s
kubectl rollout status -n kube-system deployment/fluence-webhook --timeout=180s
kubectl rollout status -n kube-system daemonset/fluence-deviceplugin --timeout=180s
kubectl get pods -n kube-system | grep fluence
```
```console
fluence-6d9b7c8f4-x2k9p             1/1     Running   0          30s
fluence-deviceplugin-4jq8z          1/1     Running   0          30s
fluence-deviceplugin-9xm2k          1/1     Running   0          30s
fluence-deviceplugin-tr5wp          1/1     Running   0          30s
fluence-webhook-7f8c6d5b9-q4m2n     1/1     Running   0          30s
```

This is quick because `kind-up` pulled the fluence image once on your machine and loaded it into
all three nodes. Without that, every node pulls the image from the registry on its own, and a
DaemonSet on three nodes means three pulls of an image that carries flux-sched.

For interested folks, the first manifest has a service, RBACs, a `KubeSchedulerConfiguration` that enables the plugin, the scheduler deployment, a mutating webhook that injects environment into pods, and a priority class. The second is a ConfigMap describing two quantum backends and a DaemonSet, the device plugin, that advertises them.


### Fluence Graph

Let's take a look at fluence logs to see what it is doing.  The first thing the scheduler logs is the graph it discovered. Each of the three nodes is a vertex with its cores and memory under it:

```bash
kubectl logs -n kube-system deployment/fluence | grep -A 40 "RESOURCE GRAPH" | head -60
```

`knownDevices=[qdevice qpu qubit]` in the header are the resource types from the config, modeled
in the graph alongside the nodes. Without the resources config it would say `knownDevices=[]` and
the graph would be the three nodes only.

<div class="alert alert-block" style="background-color:#DCECF4;color:#06293D">
<span style="font-weight:600">Tip:</span> The fluence logs will give you useful information about requests, including the graph at startup, jobspecs and allocations. When you need to debug, this is the first place to look.
</div>

## 4. Schedule a Gang

A gang is a set of pods that must start together or not at all. Kubernetes 1.36 has a native API for it, the `PodGroup`. Fluence also uses PodGroups for gang scheduling. Let's test creating a simple gang with [`manifests/gang.yaml`](manifests/gang.yaml).

```bash
kubectl apply -f manifests/gang.yaml
kubectl get podgroups.scheduling.k8s.io
kubectl get pods -l app=training -o wide
```

Each pod is on a worker, and the group says `Scheduled`. In the scheduler log the group is one match: a jobspec with two slots of one core each. Let's look at the jobspec:

```bash
kubectl logs -n kube-system deployment/fluence 
```

Now the same gang asking for more than any node has, [`manifests/gang-too-big.yaml`](manifests/gang-too-big.yaml),
five hundred cores per pod:

```bash
kubectl apply -f manifests/gang-too-big.yaml
sleep 10
kubectl get podgroups.scheduling.k8s.io
kubectl get pods -l app=toobig
```

```console
NAME       POLICY   WORKLOAD   STATUS      AGE
toobig     Gang     <none>     Pending     10s
training   Gang     <none>     Scheduled   2m

NAME                      READY   STATUS    RESTARTS   AGE
toobig-7c6b5d4a3-klmno    0/1     Pending   0          10s
toobig-7c6b5d4a3-pqrst    0/1     Pending   0          10s
```

`Pending` only says the pods are not bound to a node. Did they go through fluence at all? Yes,
and they keep going through it. Every scheduling cycle the scheduler picks them up, fluence's
PreFilter generates the group's jobspec and asks Fluxion to match it, Fluxion finds no node with
five hundred cores, and fluence returns `Unschedulable`. The scheduler records that on the pod as
an event and retries later with backoff. The pods are in the scheduler's queue the whole time,
they are just never placed. The pod's events show the verdict and who gave it:

```bash
kubectl describe pod -l app=toobig | grep -A 4 "^Events"
```
```console
Events:
  Type     Reason            Age   From     Message
  ----     ------            ----  ----     -------
  Warning  FailedScheduling  12s   fluence  0/3 nodes are available: fluxion match failed: ...
```

`From: fluence` is the point. The default scheduler never saw these pods. And in the fluence log
the attempt is there too, as a match that failed:

```bash
kubectl logs -n kube-system deployment/fluence | grep -E "MATCH.*FAILED|no allocation" | tail -3
```

<div class="alert alert-block" style="background-color:#DCECF4;color:#06293D">
<span style="font-weight:600">Notice:</span> Nothing is bound, not one pod of the two. The difference from a default scheduler shows when only one of them would fit: fluence still binds neither. A gang is matched against the graph as a unit, so there is never half a gang holding a node while the other half waits. That is the same idea as Module 1's held classical job, applied to a set of pods.
</div>

Deleting a `PodGroup` or a pod frees its Fluxion allocation, which the log reports as a cancel. If a `PodGroup` lingers in `Terminating`, it is waiting on the `podgroup-protection`
finalizer, and the second command clears it.

```bash
kubectl delete -f manifests/gang-too-big.yaml
kubectl patch podgroup toobig --type=merge -p '{"metadata":{"finalizers":null}}' 2>/dev/null || true
kubectl logs -n kube-system deployment/fluence | grep -i cancel | tail -3
```

I (@vsoch) needed the second command frequently during development!

## 5. Modeled resources: a QPU as a vertex in the graph

In Module 1 the device vertices in the Flux graph carried no custom resources. We were just scheduling gangs. Now we want to tell fluence about our custom resources, the quantum devices. Take a look at [`manifests/fluence-resources.yaml`](manifests/fluence-resources.yaml), which you applied in section 3. It defines the resources and the device plugin in one file. 

<details>

<summary>How does it work?</summary>

The ConfigMap holds a `resources.yaml` that describes two backends, `ibm_fez` and `ibm_marrakesh`, each a `qdevice` with one `qpu` holding 156 `qubit` children, and an attribute set, `vendor: ibm` and `qrmi_type: qiskit-runtime-service`, shared by both. Nothing in it is quantum specific to fluence: a type is a string, a resource is a tree with counts, and attributes are a map. Three processes read that one document. The **scheduler** builds the backends into its graph as vertices with the attributes as filterable properties. The **device plugin**, the DaemonSet in the same file, advertises each type as an extended resource, `fluxion.flux-framework.org/qpu` and so on, so that a pod requesting one passes Kubernetes' own admission. The fluence **webhook** derives the environment it will inject, `FLUXION_BACKEND` plus one `FLUXION_<KEY>` per attribute key. Upstream fluence ships the ConfigMap and the DaemonSet as two files because the device plugin is optional in its design. Here they are one step, so they are one file.

</details>

The nodes advertise the types:

```bash
kubectl get nodes -o jsonpath='{range .items[*]}{.metadata.name}{"\t"}{.status.allocatable}{"\n"}{end}' | grep -o 'fluxion[^,]*' | sort -u
```
```console
fluxion.flux-framework.org/qdevice":"1000
fluxion.flux-framework.org/qpu":"1000
fluxion.flux-framework.org/qubit":"1000
```

<div class="alert alert-block" style="background-color:#91C2D8;color:#06293D">
<span style="font-weight:600">Description:</span> The thousand is a ceiling, not a count. The device plugin only exists so that a pod asking for a <code>qpu</code> is admitted by Kubernetes' own fit check, since a backend is reachable from any node. The real gate is the graph: two backends, one exclusive <code>qpu</code> each, which is what <code>knownDevices</code> in the graph header was about.
</div>

Now three pods that each want one `qpu`, [`manifests/qpu-pods.yaml`](manifests/qpu-pods.yaml).
None of them names a backend. Each just requests `fluxion.flux-framework.org/qpu: "1"` and echoes
the environment it was given. busybox stands in for a workload, so there are no credentials here.

```bash
kubectl apply -f manifests/qpu-pods.yaml
sleep 10
kubectl get pods -l app=sampler -o custom-columns='NAME:.metadata.name,STATUS:.status.phase,NODE:.spec.nodeName,BACKEND:.metadata.annotations.fluence\.flux-framework\.org/backend'
```
```console
NAME        STATUS    NODE              BACKEND
sampler-1   Running   fluence-worker    ibm_fez
sampler-2   Running   fluence-worker2   ibm_marrakesh
sampler-3   Pending   <none>            <none>
```

Two backends, two placed pods, and the third waits. Nothing was configured on the pods to make
that happen: fluence matched each request against the graph, took an exclusive `qpu` for each of
the first two, found none left for the third, and stamped the chosen backend on the pod as an
annotation in PreBind. The webhook had already, at admission, injected downward-API environment
variables that read that annotation, so the container sees its backend without the pod having
named one:

```bash
kubectl logs sampler-1
kubectl logs sampler-2
kubectl get pod sampler-1 -o jsonpath='{range .spec.containers[0].env[*]}{.name}{"\n"}{end}'
```
```console
BACKEND=ibm_fez VENDOR=ibm
BACKEND=ibm_marrakesh VENDOR=ibm
FLUXION_BACKEND
FLUXION_QRMI_TYPE
FLUXION_VENDOR
```

<div class="alert alert-block" style="background-color:#DCECF4;color:#06293D">
<span style="font-weight:600">Notice:</span> Compare with Module 1. There, the user chose the device and exported its four QRMI variables, and the scheduler's graph only enforced a quota. Here the devices and their attributes are modeled, the scheduler selects, and the workload learns what it got from <code>FLUXION_BACKEND</code>. What has not moved is the credential: a real workload in place of busybox would mount its api key from a Secret on the pod, and the scheduler never sees it. The same separation as the pair in Module 1.
</div>

Delete one of the placed pods and the waiting one takes its backend. The deletion cancels the
Fluxion allocation, and the next scheduling cycle finds a free `qpu`:

```bash
kubectl delete pod sampler-1
sleep 10
kubectl get pods -l app=sampler -o custom-columns='NAME:.metadata.name,STATUS:.status.phase,BACKEND:.metadata.annotations.fluence\.flux-framework\.org/backend'
```
```console
NAME        STATUS    BACKEND
sampler-2   Running   ibm_marrakesh
sampler-3   Running   ibm_fez
```

To add a backend, or an attribute to filter on, edit the ConfigMap and restart the scheduler and
the webhook, `kubectl rollout restart -n kube-system deployment/fluence deployment/fluence-webhook`.
They read the config only at startup. The device plugin and the webhook derive what they advertise
and inject from the same document, so they cannot drift from the graph.


## 4. Clean up

Remove the workloads first, then the cluster. The cluster lives on your docker, so it outlives
this container unless you delete it:

```bash
kubectl delete -f manifests/qpu-pods.yaml --wait=false
kubectl delete -f manifests/gang.yaml
kubectl patch podgroup training --type=merge -p '{"metadata":{"finalizers":null}}' 2>/dev/null || true
kind-down
```

<div style="background:#DCECF4;border-left:6px solid #D9A441;padding:12px 18px;color:#06293D"><strong>Module 2 complete</strong></div>
